In [ ]:
if category:
    print("\n" + "=" * 70)
    print("ANALYSIS SUMMARY")
    print("=" * 70)
    
    coverage = 100 * stats['sentences_with_triplets'] / stats['total_sentences']
    print(f"\n1. DATA COVERAGE")
    print(f"   - {coverage:.1f}% of sentences have triplet annotations")
    print(f"   - {stats['sentences_without_triplets']} sentences lack triplets")
    
    # Sentiment analysis
    sentiments = stats['sentiment_distribution']
    total = sum(sentiments.values())
    pos_pct = 100 * sentiments[2] / max(total, 1)
    neg_pct = 100 * sentiments[0] / max(total, 1)
    
    print(f"\n2. SENTIMENT ANALYSIS")
    print(f"   - Positive triplets: {pos_pct:.1f}%")
    print(f"   - Negative triplets: {neg_pct:.1f}%")
    if pos_pct > neg_pct:
        print(f"   - Overall: More positive than negative (diff: {pos_pct - neg_pct:.1f}%)")
    else:
        print(f"   - Overall: More negative than positive (diff: {neg_pct - pos_pct:.1f}%)")
    
    # Aspect analysis
    top_aspect = list(stats['aspect_frequency'].items())[0] if stats['aspect_frequency'] else ('N/A', 0)
    print(f"\n3. KEY ASPECTS")
    print(f"   - Most discussed: {top_aspect[0]} ({top_aspect[1]} mentions)")
    print(f"   - Unique aspects: {len(stats['aspect_frequency'])}")
    
    # Quality metrics
    print(f"\n4. ANNOTATION QUALITY")
    print(f"   - Avg triplets per sentence: {stats['avg_triplets_per_sentence']:.2f}")
    print(f"   - Max triplets in single sentence: {stats['max_triplets_per_sentence']}")
    print(f"   - Sentences with multiple triplets: {stats['sentences_with_multiple_triplets']}")
    
    print("\n" + "=" * 70)

## Summary and Insights

Key findings and recommendations based on the labeled data analysis.

In [ ]:
if category:
    sentiment_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    
    print("ANNOTATED SENTENCE EXAMPLES")
    print("=" * 70)
    
    # Show examples with all sentiments
    for target_sentiment in [0, 1, 2]:
        df_filtered = filter_by_sentiment(df, target_sentiment)
        if len(df_filtered) > 0:
            print(f"\n{sentiment_names[target_sentiment].upper()} SENTIMENT EXAMPLES")
            print("-" * 70)
            
            sample = df_filtered.sample(min(2, len(df_filtered))).reset_index(drop=True)
            for idx, row in sample.iterrows():
                print(f"\nExample {idx+1}:")
                print(f"  Sentence: {row['sentence_text']}")
                print(f"  Rating: {row['rating']}/5")
                print(f"  Triplets:")
                for triplet in row['triplets']:
                    print(f"    - Aspect: {triplet['aspect']}")
                    print(f"      Opinion: {triplet['opinion']}")
                    print(f"      Sentiment: {triplet['sentiment']} ({sentiment_names[triplet['sentiment']]})")

## Sample Annotated Sentences

Display detailed examples of annotated sentences with different characteristics.

In [ ]:
if category:
    # Example 1: Get positive SENTENCES
    df_positive = filter_by_sentiment(df, 2)
    print(f"Positive sentences: {len(df_positive)}/{len(df)}")
    print("\nSample positive sentences:")
    for idx in range(min(3, len(df_positive))):
        row = df_positive.iloc[idx]
        print(f"\n  Sentence: {row['sentence_text'][:100]}...")
        print(f"  Triplets: {row['triplets']}")
    
    # Example 2: Get sentences mentioning specific aspect
    df_battery = filter_by_aspect(df, 'battery life', case_sensitive=False)
    print(f"\n\nSentences mentioning 'battery life': {len(df_battery)}/{len(df)}")
    
    # Example 3: Export subset to JSON
    output_file = f'../outputs/reports/positive_sentences_{category}.json'
    export_to_json(df_positive.head(100), output_file)
    
    # Example 4: Generate QA report
    qa_report_file = f'../outputs/reports/qa_report_{category}.txt'
    qa_results = generate_qa_report(df, qa_report_file)
    print(f"\nGenerated QA report at {qa_report_file}")

## Data Filtering Examples

Examples of filtering and exporting data.

In [ ]:
if category:
    # Get aspect-sentiment matrix
    matrix = get_aspect_sentiment_matrix(df)
    
    print("ASPECT-SENTIMENT MATRIX (Top 15 aspects)")
    print("=" * 70)
    print(matrix.head(15).to_string())
    
    # Calculate sentiment bias for each aspect
    sentiment_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    
    print("\n\nASPECT SENTIMENT BIAS (Dominant sentiment per aspect)")
    print("=" * 70)
    print(f"{'Aspect':<30} {'Dominant Sentiment':<15} {'Count':<8}")
    print("-" * 70)
    
    for aspect in matrix.head(15).index:
        row = matrix.loc[aspect]
        dominant_sentiment = row.idxmax()
        count = int(row.max())
        sentiment_name = sentiment_names[dominant_sentiment]
        print(f"{aspect[:30]:<30} {sentiment_name:<15} {count:<8}")

## Aspect-Sentiment Analysis

Analyze the relationship between aspects and sentiments.

In [ ]:
if category:
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Sentiment distribution pie chart
    sentiments = stats['sentiment_distribution']
    labels = ['Negative (0)', 'Neutral (1)', 'Positive (2)']
    colors = ['#d62728', '#ff7f0e', '#2ca02c']
    axes[0, 0].pie(sentiments.values(), labels=labels, colors=colors, autopct='%1.1f%%')
    axes[0, 0].set_title('Sentiment Distribution')
    
    # 2. Triplets per sentence distribution
    triplet_dist = stats['triplets_per_sentence_distribution']
    axes[0, 1].bar(triplet_dist.keys(), triplet_dist.values(), color='steelblue')
    axes[0, 1].set_xlabel('Number of Triplets per Sentence')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Distribution of Triplets per Sentence')
    
    # 3. Top 10 aspects
    top_aspects = list(stats['aspect_frequency'].items())[:10]
    aspects = [a[0][:15] for a in top_aspects]  # Truncate for display
    aspect_counts = [a[1] for a in top_aspects]
    axes[1, 0].barh(aspects, aspect_counts, color='coral')
    axes[1, 0].set_xlabel('Count')
    axes[1, 0].set_title('Top 10 Aspects')
    axes[1, 0].invert_yaxis()
    
    # 4. Top 10 opinions
    top_opinions = list(stats['opinion_frequency'].items())[:10]
    opinions = [o[0][:15] for o in top_opinions]  # Truncate for display
    opinion_counts = [o[1] for o in top_opinions]
    axes[1, 1].barh(opinions, opinion_counts, color='lightblue')
    axes[1, 1].set_xlabel('Count')
    axes[1, 1].set_title('Top 10 Opinions')
    axes[1, 1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(f'../outputs/reports/analysis_visualization_{category}.png', dpi=100, bbox_inches='tight')
    print(f"Saved visualization to ../outputs/reports/analysis_visualization_{category}.png")
    plt.show()

## Visualization

Create visualizations of the data distribution.

In [ ]:
if category:
    # Get comprehensive statistics
    stats = get_triplet_statistics(df)
    
    print("\nTRIPLET STATISTICS")
    print("=" * 50)
    print(f"Total sentences: {stats['total_sentences']}")
    print(f"Sentences with triplets: {stats['sentences_with_triplets']}")
    print(f"Sentences without triplets: {stats['sentences_without_triplets']}")
    print(f"Coverage: {100*stats['sentences_with_triplets']/stats['total_sentences']:.1f}%")
    print(f"\nTotal triplets: {stats['total_triplets']}")
    print(f"Avg triplets/sentence: {stats['avg_triplets_per_sentence']:.2f}")
    print(f"Max triplets/sentence: {stats['max_triplets_per_sentence']}")
    print(f"Sentences with multiple triplets: {stats['sentences_with_multiple_triplets']}")
    
    print("\nSENTIMENT DISTRIBUTION")
    print("=" * 50)
    sentiments = stats['sentiment_distribution']
    total = sum(sentiments.values())
    for sentiment, name in [(0, "Negative"), (1, "Neutral"), (2, "Positive")]:
        count = sentiments[sentiment]
        pct = 100 * count / max(total, 1)
        print(f"{name:10s} ({sentiment}): {count:6d} ({pct:5.1f}%)")
    
    print("\nTOP 10 ASPECTS")
    print("=" * 50)
    for aspect, count in list(stats['aspect_frequency'].items())[:10]:
        print(f"  {aspect:30s} - {count:4d} occurrences")
    
    print("\nTOP 10 OPINIONS")
    print("=" * 50)
    for opinion, count in list(stats['opinion_frequency'].items())[:10]:
        print(f"  {opinion:30s} - {count:4d} occurrences")

## Triplet Statistics

Calculate comprehensive statistics.

In [ ]:
if category:
    # Validate triplets
    validation_results = validate_triplets(df)
    
    print("VALIDATION RESULTS")
    print("=" * 50)
    print(f"Total rows: {validation_results['total_rows']}")
    print(f"Valid rows: {validation_results['valid_rows']}")
    print(f"Invalid rows: {validation_results['invalid_rows']}")
    print(f"Validity rate: {100*validation_results['valid_rows']/validation_results['total_rows']:.1f}%")
    
    if validation_results['errors']:
        print(f"\nFound {len(validation_results['errors'])} errors:")
        for error in validation_results['errors'][:5]:
            print(f"  Row {error['row']}: {error['error']}")

## Data Validation

Validate triplet format and integrity.

In [ ]:
# Load labeled data from first category
# Replace 'YOUR_CATEGORY' with actual category name
category = input("Enter category name to analyze (e.g., 'B08QN9H9C4'): ").strip()

if not category:
    # Use first available file as default
    import glob
    labeled_files = glob.glob('../data/processed/labeled_data_*.parquet')
    if labeled_files:
        category = labeled_files[0].split('labeled_data_')[-1].replace('.parquet', '')
        print(f"Using default category: {category}")
    else:
        print("No labeled data files found. Please run hard_labeling_pipeline first.")
        category = None

if category:
    data_path = f'../data/processed/labeled_data_{category}.parquet'
    df = load_labeled_data(data_path)
    print(f"Loaded {len(df)} records for category: {category}")
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst few rows:")
    print(df.head())

## Load and Explore Data

Load labeled data and generate basic statistics.

In [ ]:
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

# Add scripts directory to path
sys.path.insert(0, '../scripts')

from labeled_data_utils import (
    load_labeled_data,
    validate_triplets,
    get_triplet_statistics,
    filter_by_sentiment,
    filter_by_aspect,
    get_aspect_sentiment_matrix,
    generate_qa_report,
    export_to_json,
    export_to_csv
)

print("Utility functions imported successfully!")

# Analysis of Hard-Labeled Triplet Data

This notebook demonstrates how to analyze and work with the aspect-opinion-sentiment triplet labeled data using utility functions.